In [1]:
import pandas as pd
import numpy as np
import pickle

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

In [2]:
ratings = pd.read_csv("../ml-25m/ratings.csv")

ratings = ratings.sample(
    n=100000,
    random_state=42
)

print(ratings.shape)

(100000, 4)


In [3]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_map = {
    user: idx
    for idx, user in enumerate(user_ids)
}

movie_map = {
    movie: idx
    for idx, movie in enumerate(movie_ids)
}

ratings["user_idx"] = ratings["userId"].map(user_map)
ratings["movie_idx"] = ratings["movieId"].map(movie_map)

print(ratings.head())

          userId  movieId  rating   timestamp  user_idx  movie_idx
15347762   99476   104374     3.5  1467897440         0          0
16647840  107979     2634     4.0   994007728         1          1
23915192  155372     1614     3.0  1097887531         2          2
10052313   65225     7153     4.0  1201382275         3          3
12214125   79161      500     5.0  1488915363         4          4


In [4]:
R = csr_matrix(
    (
        ratings["rating"],
        (ratings["user_idx"], ratings["movie_idx"])
    )
)

print(R.shape)

(55057, 10262)


In [5]:
k = 50

U, sigma, Vt = svds(R, k=k)

sigma = np.diag(sigma)

print(U.shape)
print(sigma.shape)
print(Vt.shape)

(55057, 50)
(50, 50)
(50, 10262)


In [6]:
user_index = 0

user_prediction = np.dot(
    np.dot(U[user_index], sigma),
    Vt
)

top10 = np.argsort(user_prediction)[::-1][:10]

print(top10)

[599 756 397 423  21 573 245 362 124 583]


In [7]:
movies = pd.read_csv("../ml-25m/movies.csv")

movie_ids = list(movie_map.keys())

recommended_ids = [
    movie_ids[i]
    for i in top10
]

recommendations = movies[
    movies["movieId"].isin(recommended_ids)
]

recommendations[["movieId", "title", "genres"]]

,movieId,title,genres
359,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
475,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
585,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
587,595,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX
1164,1193,One Flew Over the Cuckoo's Nest (1975),Drama
1179,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
3479,3578,Gladiator (2000),Action|Adventure|Drama
4122,4226,Memento (2000),Mystery|Thriller
5840,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy


In [8]:
model = {
    "U": U,
    "sigma": sigma,
    "Vt": Vt,
    "user_map": user_map,
    "movie_map": movie_map
}

with open("../svd_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


# Day 5 — Model Evaluation & Model Persistence

## Objective
Evaluate the trained SVD recommendation model and save the trained artifacts for later use.

## Work Completed

- Generated recommendations using the trained SVD model.
- Converted predicted movie indices back to original MovieLens movie IDs.
- Retrieved movie titles and genres from the MovieLens dataset.
- Displayed Top-10 movie recommendations for a sample user.
- Saved the trained SVD model (U, Σ, Vᵀ).
- Saved sample recommendations as a CSV file for verification.

## Files Generated

- svd_model.pkl
- sample_recommendations.csv

## Outcome

The collaborative filtering model successfully generates personalized movie recommendations. The trained model and sample outputs are now stored and ready for use in later stages of the hybrid recommender system.